# 3.2 Lab: MHA vs MQA vs GQA Memory Comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.2_mqa_gqa/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.2_mqa_gqa/lab.ipynb)

This lab computes and visualizes KV cache memory for MHA, GQA, and MQA across different model configurations. Change the parameters at the top of each cell and re-run to explore.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === PARAMETERS (change these and re-run) ===
N_HEADS = 32          # number of query heads
HEAD_DIM = 128        # dimension per head
N_LAYERS = 32         # transformer layers
DTYPE_BYTES = 2       # 2 = FP16, 1 = INT8
SEQ_LEN = 4096        # context length in tokens
GPU_MEMORY_GB = 80    # total GPU memory
MODEL_WEIGHT_GB = 16  # memory used by model weights
ACTIVATION_GB = 4     # memory used by activations

## Experiment 1: KV Cache Per Token Across Variants

In [ ]:
def kv_per_token_bytes(n_kv_heads):
    """Compute KV cache bytes per token for given number of KV heads."""
    # Formula: 2 (K+V) x n_kv_heads x head_dim x n_layers x dtype_bytes
    return 2 * n_kv_heads * HEAD_DIM * N_LAYERS * DTYPE_BYTES

# Compute for each variant
variants = {
    "MHA": N_HEADS,        # all heads independent
    "GQA-8": 8,            # 8 KV head groups
    "GQA-4": 4,            # 4 KV head groups
    "GQA-2": 2,            # 2 KV head groups
    "MQA": 1,              # single shared KV
}

# Calculate memory per token in KB
kv_per_token_kb = {name: kv_per_token_bytes(kv_h) / 1024 for name, kv_h in variants.items()}

# Plot: bar chart of memory per token
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#ffe4e6", "#dbeafe", "#dcfce7", "#fef3c7", "#f3e8ff"]
bars = ax.bar(kv_per_token_kb.keys(), kv_per_token_kb.values(), color=colors, edgecolor="#000", linewidth=1.2)
# Add value labels on each bar
for bar, val in zip(bars, kv_per_token_kb.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f"{val:.0f} KB", ha="center", fontsize=10)
ax.set_ylabel("KV Cache per Token (KB)")
ax.set_title(f"KV Cache Memory per Token ({N_HEADS} query heads, {N_LAYERS} layers, FP16)")
ax.set_ylim(0, max(kv_per_token_kb.values()) * 1.15)
plt.tight_layout()
plt.show()

## Experiment 2: Concurrent Users vs Attention Variant

In [ ]:
def max_concurrent_users(n_kv_heads):
    """How many users fit in available GPU memory at SEQ_LEN context."""
    # Available memory for KV cache
    available_bytes = (GPU_MEMORY_GB - MODEL_WEIGHT_GB - ACTIVATION_GB) * (1024**3)
    # KV cache per user = per_token * seq_len
    per_user_bytes = kv_per_token_bytes(n_kv_heads) * SEQ_LEN
    return int(available_bytes / per_user_bytes)

# Compute concurrent users for each variant
users = {name: max_concurrent_users(kv_h) for name, kv_h in variants.items()}

# Plot: horizontal bar chart
fig, ax = plt.subplots(figsize=(8, 4))
names = list(users.keys())
values = list(users.values())
colors = ["#ffe4e6", "#dbeafe", "#dcfce7", "#fef3c7", "#f3e8ff"]
bars = ax.barh(names, values, color=colors, edgecolor="#000", linewidth=1.2)
# Label each bar with user count
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, str(val), va="center", fontsize=10)
ax.set_xlabel("Max Concurrent Users")
ax.set_title(f"Concurrent Users at {SEQ_LEN} tokens on A100-{GPU_MEMORY_GB}GB")
plt.tight_layout()
plt.show()

## Experiment 3: Memory Scaling with Context Length

In [ ]:
# Sweep context lengths from 512 to 128K
context_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]

fig, ax = plt.subplots(figsize=(9, 5))
line_colors = {"MHA": "#991b1b", "GQA-8": "#2563eb", "GQA-4": "#166534", "MQA": "#7c3aed"}

for name, n_kv in [("MHA", N_HEADS), ("GQA-8", 8), ("GQA-4", 4), ("MQA", 1)]:
    # Memory per sequence in MB across context lengths
    mem_mb = [kv_per_token_bytes(n_kv) * ctx / (1024**2) for ctx in context_lengths]
    ax.plot(context_lengths, mem_mb, marker="o", label=name, color=line_colors[name], linewidth=2)

# Draw available memory line
available_mb = (GPU_MEMORY_GB - MODEL_WEIGHT_GB - ACTIVATION_GB) * 1024
ax.axhline(y=available_mb, color="#64748b", linestyle="--", label=f"Available ({available_mb/1024:.0f} GB)")

ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("Context Length (tokens)")
ax.set_ylabel("KV Cache per User (MB)")
ax.set_title("KV Cache Growth with Context Length (single user)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Experiment 4: Cost Efficiency (Revenue per GPU)

In [ ]:
# === PARAMETERS ===
GPU_COST_PER_HOUR = 3.50   # A100-80GB on-demand price (USD)
TOKENS_PER_REQUEST = 500   # average output tokens per request
PRICE_PER_1M_TOKENS = 2.0  # revenue per 1M output tokens

def revenue_per_gpu_hour(n_kv_heads):
    """Estimate revenue per GPU-hour based on concurrent capacity."""
    # Max concurrent users
    n_users = max_concurrent_users(n_kv_heads)
    # Tokens generated per second (assume 40 tok/s per user decode)
    tokens_per_sec = n_users * 40
    # Tokens per hour
    tokens_per_hour = tokens_per_sec * 3600
    # Revenue per hour
    revenue = (tokens_per_hour / 1_000_000) * PRICE_PER_1M_TOKENS
    return revenue

# Compute revenue and profit for each variant
results = {}
for name, n_kv in variants.items():
    rev = revenue_per_gpu_hour(n_kv)
    profit = rev - GPU_COST_PER_HOUR
    results[name] = {"revenue": rev, "profit": profit, "users": max_concurrent_users(n_kv)}

# Plot: revenue comparison
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#ffe4e6", "#dbeafe", "#dcfce7", "#fef3c7", "#f3e8ff"]
revenues = [results[n]["revenue"] for n in variants]
bars = ax.bar(variants.keys(), revenues, color=colors, edgecolor="#000", linewidth=1.2)
ax.axhline(y=GPU_COST_PER_HOUR, color="#991b1b", linestyle="--", label=f"GPU cost (${GPU_COST_PER_HOUR}/hr)")
for bar, val in zip(bars, revenues):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"${val:.1f}", ha="center", fontsize=10)
ax.set_ylabel("Revenue per GPU-hour ($)")
ax.set_title("Revenue Impact of Attention Variant Choice")
ax.legend()
plt.tight_layout()
plt.show()

# Print summary table
print(f"{'Variant':<8} {'Users':>6} {'Rev/hr':>8} {'Profit/hr':>10}")
print("-" * 36)
for name, r in results.items():
    print(f"{name:<8} {r['users']:>6} ${r['revenue']:>7.1f} ${r['profit']:>9.1f}")

## Key Takeaways

1. GQA-8 provides 4x memory compression with negligible quality loss, making it the production default.
2. The concurrent user gain (9 to 39 on A100-80GB) translates directly to 4x revenue per GPU.
3. At long contexts (32K+), even GQA-8 becomes memory-bound, requiring KV cache quantization or eviction.
4. MQA offers 32x compression but measurably hurts multi-hop reasoning quality.